# Collections XGBoost Minimum Payment Model

Standalone notebook for Anaconda Navigator.

Goal: train an XGBoost model that predicts whether a user will pay at least the minimum payment by `limit_payment_date_mx`.

Label logic with decimal tolerance in favor of the user:

```text
paid_minimum_by_due_date = 1 if total_paid_user_level + 0.2 >= min_payment_amount
```

The model output is:

```text
probability_pay_minimum
probability_not_pay_minimum
```

This notebook reads monthly CSV files from a folder and includes model metrics, confusion matrix, F1 score, ROC/PR curves, calibration, deciles, and XGBoost feature importance by weight, gain, and cover.

## 0. Install Packages If Needed

Run this cell only if your Anaconda environment does not already have these packages.

In [4]:
# Uncomment if needed:
# %pip install pandas numpy scikit-learn xgboost matplotlib joblib shap

## 1. Imports

In [1]:
from __future__ import annotations

import json
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    mean_squared_error,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)
plt.style.use("default")

## 2. Parameters

The notebook is already pointed at your Desktop folder:

```text
/Users/juan.mendoza/Desktop/pre_collections_model
```

It will read every file matching `collections_model_*.csv`.

In [3]:
DATA_FOLDER = Path("/Users/juan.mendoza/Desktop/pre_collections_model")
FILE_PATTERN = "collections_model_*.csv"

# Optional fallback if you ever want to run one CSV instead of a folder. Leave as None for monthly files.
CSV_PATH = None

TRAINING_START_DATE = "2024-01-01"
TRAINING_END_EXCLUSIVE = "2026-04-01"

# Latest closed month = test; previous N months = validation.
VALIDATION_MONTHS = 2
TEST_MONTHS = 1

# Decimal tolerance in favor of the user.
LABEL_TOLERANCE = 0.2

# Governance switch. Compare True vs False before production.
INCLUDE_AGE = True

# Keep False for a probability model. Try True only if non-payment ranking is too weak.
USE_CLASS_WEIGHT = False

# If memory becomes a problem, sample training rows after the time split.
# Keep validation and test full.
TRAIN_SAMPLE_FRAC = 1.0
TRAIN_MAX_ROWS = None  # example: 1_500_000

# Optional emergency loader sampling. Leave as None for real evaluation.
# If set to 0.30, each CSV is sampled while loading, including validation/test months.
LOAD_SAMPLE_FRAC = None

CHUNKSIZE = 250_000
DEFAULT_NON_PAYMENT_THRESHOLD = 0.50
MODEL_VERSION = "collections_xgb_min_payment_v1_local"
OUTPUT_DIR = Path("/Users/juan.mendoza/Desktop/collections_xgboost_outputs")
RANDOM_STATE = 42

## 3. Feature Lists

In [4]:
ID_COLUMNS = ["user_id"]
DATE_COLUMNS = [
    "cutoff_date_mx",
    "snapshot_date",
    "limit_payment_date_mx",
    "last_billing_date_mx",
]

CATEGORICAL_FEATURES = [
    "payment_configuration_setting",
    "type_of_user",
]

BASE_NUMERIC_FEATURES = [
    "age",
    "vintage",
    "is_thin_file",
    "risk_band_cck",
    "risk_band_pl",
    "total_exposure",
    "credit_line",
    "proportion_of_credit_line_used",
    "min_payment_amount",
    "generate_no_fees_payment_amount",
    "total_deposits",
    "dep_minus_exp",
    "max_dpd_1m",
    "max_dpd_2m",
    "max_dpd_3m",
    "max_dpd_6m",
    "max_dpd_12m",
    "count_delinquencies",
    "paid_at_least_minimum",
    "paid_at_least_minimum_score_6m",
    "last_statement_ratio",
    "avg_payment_ratio_last_5m",
    "delta_payment_ratio",
    "change_proportion_credit_line_used_vs_1m_prior",
    "change_proportion_credit_line_used_vs_2m_prior",
    "change_proportion_credit_line_used_vs_3m_prior",
]

DERIVED_NUMERIC_FEATURES = [
    "days_until_due",
    "days_since_last_billing",
    "statement_month",
    "statement_day_of_month",
    "min_payment_to_deposit_ratio",
    "min_payment_to_credit_line_ratio",
    "exposure_to_credit_line_ratio",
    "deposits_to_exposure_ratio",
]

ALIAS_COLUMNS = [
    "statement_min_payment_amount",
    "statement_generate_no_fees_payment_amount",
    "statement_total_paid_user_level",
]

POSSIBLE_INPUT_COLUMNS = sorted(
    set(
        ID_COLUMNS
        + DATE_COLUMNS
        + CATEGORICAL_FEATURES
        + BASE_NUMERIC_FEATURES
        + ALIAS_COLUMNS
        + ["total_paid_user_level", "paid_minimum_by_due_date"]
    )
)

## 4. Helper Functions

In [5]:
def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)


def safe_divide(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    result = numerator.astype(float) / denominator.replace({0: np.nan}).astype(float)
    return result.replace([np.inf, -np.inf], np.nan)


def normalize_column_aliases(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    alias_map = {
        "statement_min_payment_amount": "min_payment_amount",
        "statement_generate_no_fees_payment_amount": "generate_no_fees_payment_amount",
        "statement_total_paid_user_level": "total_paid_user_level",
    }
    # Prefer the statement aliases because those came from pre_collections_statement.
    for alias, canonical in alias_map.items():
        if alias in df.columns:
            df[canonical] = df[alias]
    return df


def require_columns(df: pd.DataFrame, required_columns: list[str]) -> None:
    missing = sorted(set(required_columns) - set(df.columns))
    if missing:
        raise ValueError(
            "Missing required columns:\n" + "\n".join(f"- {col}" for col in missing)
        )


def get_csv_files() -> list[Path]:
    if CSV_PATH is not None:
        paths = [Path(CSV_PATH)]
    else:
        paths = sorted(DATA_FOLDER.glob(FILE_PATTERN))
    if not paths:
        raise FileNotFoundError(
            f"No CSV files found. DATA_FOLDER={DATA_FOLDER}, FILE_PATTERN={FILE_PATTERN}"
        )
    return paths


def load_csv_files(paths: list[Path]) -> pd.DataFrame:
    frames = []
    for path in paths:
        print(f"Loading {path.name} ...")
        header = pd.read_csv(path, nrows=0).columns.tolist()
        usecols = [col for col in header if col in POSSIBLE_INPUT_COLUMNS]
        if LOAD_SAMPLE_FRAC is None:
            frame = pd.read_csv(path, usecols=usecols, low_memory=False)
        else:
            chunks = []
            for chunk in pd.read_csv(
                path, usecols=usecols, chunksize=CHUNKSIZE, low_memory=False
            ):
                chunks.append(
                    chunk.sample(frac=LOAD_SAMPLE_FRAC, random_state=RANDOM_STATE)
                )
            frame = pd.concat(chunks, ignore_index=True)
        frame["source_file"] = path.name
        frames.append(frame)
    return pd.concat(frames, ignore_index=True)


def add_label_and_features(raw_df: pd.DataFrame) -> pd.DataFrame:
    df = normalize_column_aliases(raw_df)
    required = [
        "user_id",
        "snapshot_date",
        "cutoff_date_mx",
        "limit_payment_date_mx",
        "min_payment_amount",
        "total_paid_user_level",
    ]
    require_columns(df, required)

    if "paid_minimum_by_due_date" in df.columns:
        df["exported_paid_minimum_by_due_date"] = pd.to_numeric(
            df["paid_minimum_by_due_date"], errors="coerce"
        )

    for column in DATE_COLUMNS:
        if column in df.columns:
            df[column] = (
                pd.to_datetime(df[column], errors="coerce")
                .dt.tz_localize(None)
                .dt.normalize()
            )

    for column in sorted(set(BASE_NUMERIC_FEATURES + ["total_paid_user_level"])):
        if column in df.columns:
            df[column] = pd.to_numeric(df[column], errors="coerce")

    for column in CATEGORICAL_FEATURES:
        if column not in df.columns:
            df[column] = "UNKNOWN"
        df[column] = df[column].fillna("UNKNOWN").astype(str)

    df["paid_minimum_by_due_date"] = (
        df["total_paid_user_level"].fillna(0) + LABEL_TOLERANCE
        >= df["min_payment_amount"]
    ).astype(int)

    if "exported_paid_minimum_by_due_date" in df.columns:
        comparable = df["exported_paid_minimum_by_due_date"].notna()
        mismatch_rate = (
            df.loc[comparable, "exported_paid_minimum_by_due_date"].astype(int)
            != df.loc[comparable, "paid_minimum_by_due_date"]
        ).mean()
        print(f"Exported label mismatch rate vs notebook label: {mismatch_rate:.6f}")

    df["days_until_due"] = (df["limit_payment_date_mx"] - df["snapshot_date"]).dt.days
    if "last_billing_date_mx" in df.columns:
        df["days_since_last_billing"] = (
            df["snapshot_date"] - df["last_billing_date_mx"]
        ).dt.days
    else:
        df["days_since_last_billing"] = np.nan

    df["statement_month"] = df["snapshot_date"].dt.month
    df["statement_day_of_month"] = df["snapshot_date"].dt.day
    df["snapshot_month"] = df["snapshot_date"].dt.to_period("M").astype(str)

    df["min_payment_to_deposit_ratio"] = safe_divide(
        df["min_payment_amount"],
        df.get("total_deposits", pd.Series(np.nan, index=df.index)),
    )
    df["min_payment_to_credit_line_ratio"] = safe_divide(
        df["min_payment_amount"],
        df.get("credit_line", pd.Series(np.nan, index=df.index)),
    )
    df["exposure_to_credit_line_ratio"] = safe_divide(
        df.get("total_exposure", pd.Series(np.nan, index=df.index)),
        df.get("credit_line", pd.Series(np.nan, index=df.index)),
    )
    df["deposits_to_exposure_ratio"] = safe_divide(
        df.get("total_deposits", pd.Series(np.nan, index=df.index)),
        df.get("total_exposure", pd.Series(np.nan, index=df.index)),
    )
    return df


def filter_eligible_rows(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out = out[out["snapshot_date"] >= pd.Timestamp(TRAINING_START_DATE)]
    out = out[out["snapshot_date"] < pd.Timestamp(TRAINING_END_EXCLUSIVE)]
    out = out[out["snapshot_date"] == out["cutoff_date_mx"]]
    out = out[out["min_payment_amount"] > 0]
    return out.sort_values(["snapshot_date", "user_id"]).reset_index(drop=True)


def choose_time_splits(df: pd.DataFrame):
    months = sorted(df["snapshot_month"].dropna().unique())
    needed = VALIDATION_MONTHS + TEST_MONTHS + 1
    if len(months) < needed:
        raise ValueError(
            f"Need at least {needed} months. Found {len(months)}: {months}"
        )
    test_months = months[-TEST_MONTHS:]
    validation_months = months[-(TEST_MONTHS + VALIDATION_MONTHS) : -TEST_MONTHS]
    train_months = months[: -(TEST_MONTHS + VALIDATION_MONTHS)]
    return train_months, validation_months, test_months


def ks_statistic(y_binary_positive: pd.Series, scores: np.ndarray) -> float:
    positives = scores[np.array(y_binary_positive) == 1]
    negatives = scores[np.array(y_binary_positive) == 0]
    if len(positives) == 0 or len(negatives) == 0:
        return np.nan
    thresholds = np.sort(np.unique(scores))
    pos_cdf = np.searchsorted(np.sort(positives), thresholds, side="right") / len(
        positives
    )
    neg_cdf = np.searchsorted(np.sort(negatives), thresholds, side="right") / len(
        negatives
    )
    return float(np.max(np.abs(pos_cdf - neg_cdf)))


def metrics_table(
    y_pay_true: pd.Series, probability_pay: np.ndarray, split_name: str
) -> dict:
    y_nonpay_true = 1 - np.array(y_pay_true)
    probability_nonpay = 1 - probability_pay
    out = {
        "split": split_name,
        "rows": int(len(y_pay_true)),
        "actual_pay_rate": float(np.mean(y_pay_true)),
        "mean_predicted_pay_probability": float(np.mean(probability_pay)),
        "log_loss_pay": float(log_loss(y_pay_true, probability_pay, labels=[0, 1])),
        "brier_score_pay": float(brier_score_loss(y_pay_true, probability_pay)),
        "rmse_probability_pay": float(
            np.sqrt(mean_squared_error(y_pay_true, probability_pay))
        ),
        "ks_non_payment": ks_statistic(y_nonpay_true, probability_nonpay),
    }
    if len(np.unique(y_pay_true)) == 2:
        out["roc_auc_pay"] = float(roc_auc_score(y_pay_true, probability_pay))
        out["pr_auc_pay"] = float(average_precision_score(y_pay_true, probability_pay))
        out["roc_auc_non_payment"] = float(
            roc_auc_score(y_nonpay_true, probability_nonpay)
        )
        out["pr_auc_non_payment"] = float(
            average_precision_score(y_nonpay_true, probability_nonpay)
        )
    else:
        out["roc_auc_pay"] = np.nan
        out["pr_auc_pay"] = np.nan
        out["roc_auc_non_payment"] = np.nan
        out["pr_auc_non_payment"] = np.nan
    return out


def threshold_metrics(
    y_nonpay_true: pd.Series,
    probability_nonpay: np.ndarray,
    threshold: float,
    split_name: str,
) -> dict:
    y_pred = (probability_nonpay >= threshold).astype(int)
    return {
        "split": split_name,
        "threshold_non_payment": float(threshold),
        "accuracy": float(accuracy_score(y_nonpay_true, y_pred)),
        "precision_non_payment": float(
            precision_score(y_nonpay_true, y_pred, zero_division=0)
        ),
        "recall_non_payment": float(
            recall_score(y_nonpay_true, y_pred, zero_division=0)
        ),
        "f1_non_payment": float(f1_score(y_nonpay_true, y_pred, zero_division=0)),
    }


def find_best_f1_threshold(y_nonpay_true: pd.Series, probability_nonpay: np.ndarray):
    thresholds = np.linspace(0.05, 0.95, 91)
    sweep = pd.DataFrame(
        [
            threshold_metrics(y_nonpay_true, probability_nonpay, t, "validation")
            for t in thresholds
        ]
    )
    best = sweep.sort_values(
        ["f1_non_payment", "recall_non_payment"], ascending=False
    ).iloc[0]
    return float(best["threshold_non_payment"]), sweep


def make_decile_table(scored_df: pd.DataFrame) -> pd.DataFrame:
    tmp = scored_df.copy()
    tmp["risk_rank"] = tmp["probability_not_pay_minimum"].rank(
        method="first", ascending=False
    )
    tmp["risk_decile"] = (
        pd.qcut(tmp["risk_rank"], q=10, labels=False, duplicates="drop") + 1
    )
    total_nonpayers = int((1 - tmp["paid_minimum_by_due_date"]).sum())
    deciles = (
        tmp.groupby("risk_decile")
        .agg(
            users=("user_id", "count"),
            avg_predicted_pay_probability=("probability_pay_minimum", "mean"),
            avg_predicted_non_payment_probability=(
                "probability_not_pay_minimum",
                "mean",
            ),
            actual_pay_rate=("paid_minimum_by_due_date", "mean"),
            non_payers=("paid_minimum_by_due_date", lambda x: int((1 - x).sum())),
        )
        .reset_index()
        .sort_values("risk_decile")
    )
    deciles["non_payment_capture_rate"] = deciles["non_payers"] / max(
        total_nonpayers, 1
    )
    deciles["cumulative_non_payment_capture_rate"] = deciles[
        "non_payers"
    ].cumsum() / max(total_nonpayers, 1)
    return deciles


def probability_bucket(probability_pay: pd.Series) -> pd.Series:
    return pd.cut(
        probability_pay,
        bins=[-np.inf, 0.25, 0.50, 0.75, 0.90, np.inf],
        labels=[
            "very_low_pay_probability",
            "low_pay_probability",
            "medium_pay_probability",
            "high_pay_probability",
            "very_high_pay_probability",
        ],
    )


def original_feature_name(
    transformed_name: str, categorical_features: list[str]
) -> str:
    if transformed_name.startswith("num__"):
        return transformed_name.replace("num__", "", 1)
    if transformed_name.startswith("cat__"):
        body = transformed_name.replace("cat__", "", 1)
        for feature in categorical_features:
            if body == feature or body.startswith(feature + "_"):
                return feature
        return body
    return transformed_name

## 5. Load Monthly CSV Files

In [ ]:
csv_files = get_csv_files()
print(f"Found {len(csv_files)} CSV files")
for path in csv_files:
    size_mb = path.stat().st_size / (1024 * 1024)
    print(f"- {path.name}: {size_mb:,.1f} MB")

raw = load_csv_files(csv_files)
print(f"Raw rows loaded: {len(raw):,}")
print(f"Raw columns loaded: {len(raw.columns):,}")
display(raw.head())

Found 27 CSV files
- collections_model_2024_01.csv: 130.2 MB
- collections_model_2024_02.csv: 140.8 MB
- collections_model_2024_03.csv: 170.9 MB
- collections_model_2024_04.csv: 170.9 MB
- collections_model_2024_05.csv: 191.3 MB
- collections_model_2024_06.csv: 207.4 MB
- collections_model_2024_07.csv: 222.6 MB
- collections_model_2024_08.csv: 262.4 MB
- collections_model_2024_09.csv: 269.7 MB
- collections_model_2024_10.csv: 263.6 MB
- collections_model_2024_11.csv: 321.6 MB
- collections_model_2024_12.csv: 316.5 MB
- collections_model_2025_01.csv: 331.5 MB
- collections_model_2025_02.csv: 351.9 MB
- collections_model_2025_03.csv: 372.8 MB
- collections_model_2025_04.csv: 380.5 MB
- collections_model_2025_05.csv: 424.8 MB
- collections_model_2025_06.csv: 429.4 MB
- collections_model_2025_07.csv: 415.7 MB
- collections_model_2025_08.csv: 498.7 MB
- collections_model_2025_09.csv: 378.8 MB
- collections_model_2025_10.csv: 388.6 MB
- collections_model_2025_11.csv: 394.4 MB
- collections_m

## 6. Data Quality Checks

In [ ]:
df = add_label_and_features(raw)
eligible = filter_eligible_rows(df)

print(f"Eligible rows: {len(eligible):,}")
print(
    f"Snapshot range: {eligible['snapshot_date'].min().date()} to {eligible['snapshot_date'].max().date()}"
)
print(f"Months: {eligible['snapshot_month'].nunique():,}")
print(f"Paid minimum rate: {eligible['paid_minimum_by_due_date'].mean():.3f}")
print(
    f"Duplicate user_id + snapshot_date rows: {eligible.duplicated(['user_id', 'snapshot_date']).sum():,}"
)

month_summary = (
    eligible.groupby("snapshot_month")
    .agg(
        rows=("user_id", "count"),
        users=("user_id", "nunique"),
        pay_rate=("paid_minimum_by_due_date", "mean"),
        avg_min_payment=("min_payment_amount", "mean"),
        max_due_date=("limit_payment_date_mx", "max"),
    )
    .reset_index()
)
display(month_summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].plot(month_summary["snapshot_month"], month_summary["rows"], marker="o")
axes[0].set_title("Rows by snapshot month")
axes[0].set_xlabel("Snapshot month")
axes[0].set_ylabel("Rows")
axes[0].tick_params(axis="x", rotation=75)
axes[0].grid(alpha=0.25)

axes[1].plot(
    month_summary["snapshot_month"],
    month_summary["pay_rate"],
    marker="o",
    color="darkgreen",
)
axes[1].set_title("Actual pay-minimum rate by snapshot month")
axes[1].set_xlabel("Snapshot month")
axes[1].set_ylabel("Pay-minimum rate")
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis="x", rotation=75)
axes[1].grid(alpha=0.25)
plt.tight_layout()
plt.show()

missing_rate = eligible.isna().mean().sort_values(ascending=False).head(25)
fig, ax = plt.subplots(figsize=(9, 6))
missing_rate.sort_values().plot(kind="barh", ax=ax)
ax.set_title("Top missing-value rates")
ax.set_xlabel("Missing rate")
plt.tight_layout()
plt.show()

## 7. Time-Based Split

Latest month = test. Previous two months = validation. Older months = train.

In [ ]:
train_months, validation_months, test_months = choose_time_splits(eligible)
train_df = eligible[eligible["snapshot_month"].isin(train_months)].copy()
val_df = eligible[eligible["snapshot_month"].isin(validation_months)].copy()
test_df = eligible[eligible["snapshot_month"].isin(test_months)].copy()

if TRAIN_SAMPLE_FRAC < 1.0:
    train_df = train_df.sample(frac=TRAIN_SAMPLE_FRAC, random_state=RANDOM_STATE)
if TRAIN_MAX_ROWS is not None and len(train_df) > TRAIN_MAX_ROWS:
    train_df = train_df.sample(n=TRAIN_MAX_ROWS, random_state=RANDOM_STATE)

print(
    f"Train months: {train_months[0]} to {train_months[-1]} ({len(train_months)} months)"
)
print(f"Validation months: {validation_months}")
print(f"Test months: {test_months}")

split_summary = pd.DataFrame(
    [
        {
            "split": "train",
            "rows": len(train_df),
            "pay_rate": train_df["paid_minimum_by_due_date"].mean(),
        },
        {
            "split": "validation",
            "rows": len(val_df),
            "pay_rate": val_df["paid_minimum_by_due_date"].mean(),
        },
        {
            "split": "test",
            "rows": len(test_df),
            "pay_rate": test_df["paid_minimum_by_due_date"].mean(),
        },
    ]
)
display(split_summary)

## 8. Build Feature Matrix

In [ ]:
available_numeric_features = [
    feature
    for feature in BASE_NUMERIC_FEATURES + DERIVED_NUMERIC_FEATURES
    if feature in eligible.columns
]
if not INCLUDE_AGE:
    available_numeric_features = [
        feature for feature in available_numeric_features if feature != "age"
    ]

available_categorical_features = [
    feature for feature in CATEGORICAL_FEATURES if feature in eligible.columns
]
feature_columns = available_numeric_features + available_categorical_features

print(f"Numeric features: {len(available_numeric_features)}")
print(f"Categorical features: {len(available_categorical_features)}")
print(f"Total raw features: {len(feature_columns)}")
display(pd.DataFrame({"feature": feature_columns}))

X_train = train_df[feature_columns]
X_val = val_df[feature_columns]
X_test = test_df[feature_columns]

y_train_pay = train_df["paid_minimum_by_due_date"]
y_val_pay = val_df["paid_minimum_by_due_date"]
y_test_pay = test_df["paid_minimum_by_due_date"]

y_train_nonpay = 1 - y_train_pay
y_val_nonpay = 1 - y_val_pay
y_test_nonpay = 1 - y_test_pay

## 9. Train XGBoost Model

In [ ]:
numeric_transformer = Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))])
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_one_hot_encoder()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, available_numeric_features),
        ("cat", categorical_transformer, available_categorical_features),
    ]
)

X_train_t = preprocessor.fit_transform(X_train)
X_val_t = preprocessor.transform(X_val)
X_test_t = preprocessor.transform(X_test)

negative_count = int((y_train_pay == 0).sum())
positive_count = int((y_train_pay == 1).sum())
scale_pos_weight = negative_count / max(positive_count, 1) if USE_CLASS_WEIGHT else 1.0

xgb_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    n_estimators=900,
    learning_rate=0.04,
    max_depth=4,
    min_child_weight=25,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=8.0,
    reg_alpha=1.0,
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    scale_pos_weight=scale_pos_weight,
)

try:
    xgb_model.fit(
        X_train_t,
        y_train_pay,
        eval_set=[(X_train_t, y_train_pay), (X_val_t, y_val_pay)],
        early_stopping_rounds=50,
        verbose=False,
    )
except TypeError:
    xgb_model.fit(
        X_train_t,
        y_train_pay,
        eval_set=[(X_train_t, y_train_pay), (X_val_t, y_val_pay)],
        verbose=False,
    )

print("Model trained")
print(f"USE_CLASS_WEIGHT: {USE_CLASS_WEIGHT}")
print(f"scale_pos_weight: {scale_pos_weight:.3f}")
print(f"Best iteration: {getattr(xgb_model, 'best_iteration', None)}")

## 10. Training Curve

In [ ]:
try:
    evals_result = xgb_model.evals_result()
    train_logloss = evals_result["validation_0"]["logloss"]
    val_logloss = evals_result["validation_1"]["logloss"]
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(train_logloss, label="Train logloss")
    ax.plot(val_logloss, label="Validation logloss")
    ax.set_title("XGBoost training curve")
    ax.set_xlabel("Boosting round")
    ax.set_ylabel("Log loss")
    ax.legend()
    ax.grid(alpha=0.25)
    plt.show()
except Exception as exc:
    print(f"Training curve unavailable: {exc}")

## 11. Calibrate Probabilities

This converts raw XGBoost scores into better probabilities using validation predictions.

In [ ]:
val_raw_pay_prob = xgb_model.predict_proba(X_val_t)[:, 1]
calibrator = LogisticRegression(solver="lbfgs", random_state=RANDOM_STATE)
calibrator.fit(val_raw_pay_prob.reshape(-1, 1), y_val_pay)


def predict_pay_probability(raw_features: pd.DataFrame) -> np.ndarray:
    transformed = preprocessor.transform(raw_features[feature_columns])
    raw_pay_probability = xgb_model.predict_proba(transformed)[:, 1]
    return calibrator.predict_proba(raw_pay_probability.reshape(-1, 1))[:, 1]


train_pay_prob = predict_pay_probability(train_df)
val_pay_prob = predict_pay_probability(val_df)
test_pay_prob = predict_pay_probability(test_df)

train_nonpay_prob = 1 - train_pay_prob
val_nonpay_prob = 1 - val_pay_prob
test_nonpay_prob = 1 - test_pay_prob

metrics = pd.DataFrame(
    [
        metrics_table(y_train_pay, train_pay_prob, "train"),
        metrics_table(y_val_pay, val_pay_prob, "validation"),
        metrics_table(y_test_pay, test_pay_prob, "test"),
    ]
)
display(metrics)

## 12. Threshold, Confusion Matrix, F1 Score

In [ ]:
best_f1_threshold, threshold_sweep = find_best_f1_threshold(
    y_val_nonpay, val_nonpay_prob
)
print(f"Default non-payment threshold: {DEFAULT_NON_PAYMENT_THRESHOLD:.2f}")
print(f"Best validation F1 threshold: {best_f1_threshold:.2f}")

threshold_results = pd.DataFrame(
    [
        threshold_metrics(
            y_val_nonpay,
            val_nonpay_prob,
            DEFAULT_NON_PAYMENT_THRESHOLD,
            "validation_default_threshold",
        ),
        threshold_metrics(
            y_test_nonpay,
            test_nonpay_prob,
            DEFAULT_NON_PAYMENT_THRESHOLD,
            "test_default_threshold",
        ),
        threshold_metrics(
            y_val_nonpay,
            val_nonpay_prob,
            best_f1_threshold,
            "validation_best_f1_threshold",
        ),
        threshold_metrics(
            y_test_nonpay, test_nonpay_prob, best_f1_threshold, "test_best_f1_threshold"
        ),
    ]
)
display(threshold_results)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(
    threshold_sweep["threshold_non_payment"],
    threshold_sweep["precision_non_payment"],
    label="Precision",
)
ax.plot(
    threshold_sweep["threshold_non_payment"],
    threshold_sweep["recall_non_payment"],
    label="Recall",
)
ax.plot(
    threshold_sweep["threshold_non_payment"],
    threshold_sweep["f1_non_payment"],
    label="F1",
)
ax.axvline(
    best_f1_threshold,
    color="black",
    linestyle="--",
    label=f"Best F1 threshold = {best_f1_threshold:.2f}",
)
ax.set_title("Validation threshold sweep for non-payment risk")
ax.set_xlabel("Non-payment probability threshold")
ax.set_ylabel("Metric")
ax.set_ylim(0, 1)
ax.legend()
ax.grid(alpha=0.25)
plt.show()

In [ ]:
test_nonpay_pred = (test_nonpay_prob >= best_f1_threshold).astype(int)
cm = confusion_matrix(y_test_nonpay, test_nonpay_pred, labels=[0, 1])

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Actually paid", "Actually did not pay"],
)
fig, ax = plt.subplots(figsize=(7, 6))
disp.plot(cmap="Blues", values_format=",d", ax=ax, colorbar=False)
ax.set_title(f"Test confusion matrix at non-payment threshold {best_f1_threshold:.2f}")
plt.show()

print("Classification report for non-payment as the positive class:")
print(
    classification_report(
        y_test_nonpay,
        test_nonpay_pred,
        target_names=["paid", "did_not_pay"],
        zero_division=0,
    )
)

## 13. ROC and Precision-Recall Curves

Evaluated for non-payment risk.

In [ ]:
fpr, tpr, _ = roc_curve(y_test_nonpay, test_nonpay_prob)
precision, recall, _ = precision_recall_curve(y_test_nonpay, test_nonpay_prob)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(
    fpr, tpr, label=f"ROC AUC = {roc_auc_score(y_test_nonpay, test_nonpay_prob):.3f}"
)
axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray")
axes[0].set_title("Test ROC curve: non-payment")
axes[0].set_xlabel("False positive rate")
axes[0].set_ylabel("True positive rate")
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].plot(
    recall,
    precision,
    label=f"PR AUC = {average_precision_score(y_test_nonpay, test_nonpay_prob):.3f}",
)
axes[1].set_title("Test precision-recall curve: non-payment")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].legend()
axes[1].grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 14. Calibration and Decile Lift

Decile 1 is the highest non-payment risk group.

In [ ]:
test_scored = test_df[
    [
        "user_id",
        "snapshot_date",
        "cutoff_date_mx",
        "limit_payment_date_mx",
        "min_payment_amount",
        "total_paid_user_level",
        "paid_minimum_by_due_date",
        "snapshot_month",
    ]
].copy()
test_scored["probability_pay_minimum"] = test_pay_prob
test_scored["probability_not_pay_minimum"] = test_nonpay_prob
test_scored["probability_bucket"] = probability_bucket(
    test_scored["probability_pay_minimum"]
)
test_scored["model_version"] = MODEL_VERSION

decile_table = make_decile_table(test_scored)
display(decile_table)

bucket_table = (
    test_scored.groupby("probability_bucket", observed=False)
    .agg(
        users=("user_id", "count"),
        avg_predicted_pay_probability=("probability_pay_minimum", "mean"),
        actual_pay_rate=("paid_minimum_by_due_date", "mean"),
    )
    .reset_index()
)
display(bucket_table)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].plot(
    decile_table["risk_decile"],
    decile_table["avg_predicted_pay_probability"],
    marker="o",
    label="Predicted pay probability",
)
axes[0].plot(
    decile_table["risk_decile"],
    decile_table["actual_pay_rate"],
    marker="o",
    label="Actual pay rate",
)
axes[0].set_title("Calibration by risk decile")
axes[0].set_xlabel("Risk decile, 1 = highest non-payment risk")
axes[0].set_ylabel("Pay-minimum rate")
axes[0].set_ylim(0, 1)
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].bar(
    decile_table["risk_decile"],
    decile_table["cumulative_non_payment_capture_rate"],
    color="darkred",
)
axes[1].set_title("Cumulative non-payment capture by risk decile")
axes[1].set_xlabel("Risk decile included")
axes[1].set_ylabel("Cumulative capture rate")
axes[1].set_ylim(0, 1)
axes[1].grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

## 15. XGBoost Feature Importance: Weight, Gain, Cover

- **Weight**: how many times a feature is used for a split.
- **Gain**: average improvement when the feature is used.
- **Cover**: average number of rows affected by splits using the feature.

Gain and SHAP are usually more useful than weight for interpretation.

In [ ]:
transformed_feature_names = list(preprocessor.get_feature_names_out())
booster = xgb_model.get_booster()
feature_index_map = {f"f{i}": name for i, name in enumerate(transformed_feature_names)}

importance_frames = []
for importance_type in ["weight", "gain", "cover", "total_gain", "total_cover"]:
    scores = booster.get_score(importance_type=importance_type)
    frame = pd.DataFrame(
        {
            "xgb_feature": list(scores.keys()),
            importance_type: list(scores.values()),
        }
    )
    frame["transformed_feature"] = (
        frame["xgb_feature"].map(feature_index_map).fillna(frame["xgb_feature"])
    )
    frame["original_feature"] = frame["transformed_feature"].map(
        lambda x: original_feature_name(x, available_categorical_features)
    )
    grouped = frame.groupby("original_feature", as_index=False)[importance_type].sum()
    importance_frames.append(grouped)

importance_summary = importance_frames[0]
for frame in importance_frames[1:]:
    importance_summary = importance_summary.merge(
        frame, on="original_feature", how="outer"
    )
importance_summary = importance_summary.fillna(0).sort_values("gain", ascending=False)
display(importance_summary.head(30))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 7))
for ax, metric, color in zip(
    axes, ["weight", "gain", "cover"], ["steelblue", "darkgreen", "purple"]
):
    top = (
        importance_summary.sort_values(metric, ascending=False)
        .head(20)
        .sort_values(metric)
    )
    ax.barh(top["original_feature"], top[metric], color=color)
    ax.set_title(f"Top features by {metric}")
    ax.set_xlabel(metric)
plt.tight_layout()
plt.show()

## 16. Optional SHAP Explanation

This can be slower, so it samples the test set.

In [ ]:
try:
    import shap

    sample_size = min(1500, X_test_t.shape[0])
    shap_sample = X_test_t[:sample_size]
    shap_sample_dense = (
        shap_sample.toarray() if hasattr(shap_sample, "toarray") else shap_sample
    )

    explainer = shap.TreeExplainer(xgb_model)
    shap_values = explainer.shap_values(shap_sample_dense)

    shap.summary_plot(
        shap_values,
        shap_sample_dense,
        feature_names=transformed_feature_names,
        max_display=25,
    )
    shap.summary_plot(
        shap_values,
        shap_sample_dense,
        feature_names=transformed_feature_names,
        plot_type="bar",
        max_display=25,
    )
except Exception as exc:
    print(f"Skipping SHAP because it is unavailable or failed: {exc}")

## 17. Save Model Artifact and Evaluation Files

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

artifact = {
    "model_version": MODEL_VERSION,
    "preprocessor": preprocessor,
    "xgb_model": xgb_model,
    "calibrator": calibrator,
    "feature_columns": feature_columns,
    "numeric_features": available_numeric_features,
    "categorical_features": available_categorical_features,
    "label_tolerance": LABEL_TOLERANCE,
    "include_age": INCLUDE_AGE,
    "use_class_weight": USE_CLASS_WEIGHT,
    "best_f1_non_payment_threshold": best_f1_threshold,
}

joblib.dump(artifact, OUTPUT_DIR / "collections_xgboost_minimum_payment_model.joblib")
metrics.to_csv(OUTPUT_DIR / "metrics.csv", index=False)
threshold_results.to_csv(OUTPUT_DIR / "threshold_metrics.csv", index=False)
threshold_sweep.to_csv(OUTPUT_DIR / "validation_threshold_sweep.csv", index=False)
decile_table.to_csv(OUTPUT_DIR / "test_deciles.csv", index=False)
bucket_table.to_csv(OUTPUT_DIR / "test_probability_buckets.csv", index=False)
importance_summary.to_csv(
    OUTPUT_DIR / "feature_importance_weight_gain_cover.csv", index=False
)
test_scored.to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)

metadata = {
    "model_version": MODEL_VERSION,
    "data_folder": str(DATA_FOLDER),
    "file_pattern": FILE_PATTERN,
    "training_start_date": TRAINING_START_DATE,
    "training_end_exclusive": TRAINING_END_EXCLUSIVE,
    "train_months": train_months,
    "validation_months": validation_months,
    "test_months": test_months,
    "feature_columns": feature_columns,
    "label": "total_paid_user_level + 0.2 >= min_payment_amount",
    "best_f1_non_payment_threshold": best_f1_threshold,
}
with open(OUTPUT_DIR / "training_metadata.json", "w") as file:
    json.dump(metadata, file, indent=2)

print(f"Saved outputs to: {OUTPUT_DIR.resolve()}")

## 18. How to Read the Results

A good first model should show:

- Test metrics close to validation metrics. If validation is much better than test, the model is overfitting or the latest month changed.
- Lower actual pay rate in risk decile 1 than in risk decile 10.
- Calibration where predicted pay probability roughly tracks actual pay rate by decile.
- Meaningful important variables, usually prior payment behavior, delinquency history, deposits/payment capacity, utilization, risk bands, and payment setup.

Things to adjust:

- If overfitting: lower `max_depth`, increase `min_child_weight`, increase `reg_lambda`, or reduce `n_estimators`.
- If non-payers are not separated: try `USE_CLASS_WEIGHT = True`, add stronger historical payment features, or evaluate top-risk deciles instead of a fixed 0.50 threshold.
- If probabilities are poorly calibrated: improve calibration, use more validation months, or compare Platt scaling vs isotonic calibration.
- If `age` is important: rerun with `INCLUDE_AGE = False` and compare performance before production approval.

For Mage AI later, the production input should not include `total_paid_user_level` or `paid_minimum_by_due_date`; those only exist in historical training/evaluation data.

## 19. Mage AI Scoring Shape Later

Once you choose the best model, Mage can use the saved `.joblib` artifact like this:

```python
artifact = joblib.load('collections_xgboost_minimum_payment_model.joblib')
df_features = add_the_same_derived_features(current_users_dataframe)
X = df_features[artifact['feature_columns']]
X_t = artifact['preprocessor'].transform(X)
raw_pay_probability = artifact['xgb_model'].predict_proba(X_t)[:, 1]
probability_pay_minimum = artifact['calibrator'].predict_proba(raw_pay_probability.reshape(-1, 1))[:, 1]
probability_not_pay_minimum = 1 - probability_pay_minimum
```

Mage output dataframe should include:

```text
user_id
snapshot_date
cutoff_date_mx
limit_payment_date_mx
min_payment_amount
probability_pay_minimum
probability_not_pay_minimum
probability_bucket
model_version
scored_at
```